In [1]:
import os
import sys
import pandas as pd
import numpy as np
import yaml
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(os.path.join(PROJECT_ROOT, "src"))
from modeling import run_base_learner_suite
from preprocessing import split_column_types

with open(os.path.join(PROJECT_ROOT, "config", "config.yaml")) as f:
    config = yaml.safe_load(f)

SEED = config["seed"]
PROCESSED_DIR = os.path.join(PROJECT_ROOT, config["paths"]["processed_dir"])
MODELS_ROOT = os.path.join(PROJECT_ROOT, config["paths"]["models_dir"])
TABLES_DIR = os.path.join(PROJECT_ROOT, config["paths"]["tables_dir"])

MODEL_NAMES = ["dt", "rf", "gb", "svm", "nb", "knn", "xgb"]
N_TRIALS_TREE = config["optuna"]["n_trials_tree"]
N_TRIALS_SVM = config["optuna"]["n_trials_svm"]
TRIAL_BUDGET = {"dt": N_TRIALS_TREE, "rf": N_TRIALS_TREE, "gb": N_TRIALS_TREE, "xgb": N_TRIALS_TREE,
                 "svm": N_TRIALS_SVM, "knn": N_TRIALS_SVM, "nb": N_TRIALS_SVM}
SVM_FIT_ROWS = config["svm_subsample"]["fit_rows"]
N_BOOTSTRAP = config["bootstrap"]["n_resamples"]

NUMERIC_COLS = ["age_numeric", "otherChildren", "numStillBorn", "numMissCarry"]
CATEGORICAL_COLS = ["homeDistrict", "occupation", "relationshipStatus", "spouseOccupation",
                     "polygamy", "education"]

In [2]:
ug_dir = os.path.join(PROCESSED_DIR, "uganda")
train = pd.read_csv(os.path.join(ug_dir, "train.csv"))
cal = pd.read_csv(os.path.join(ug_dir, "calibration.csv"))
test = pd.read_csv(os.path.join(ug_dir, "test.csv"))

print("Districts before collapsing:", train["homeDistrict"].nunique())

RARE_THRESHOLD = 30
district_counts = train["homeDistrict"].value_counts()
keep_districts = set(district_counts[district_counts >= RARE_THRESHOLD].index)
print(f"Districts kept as own category (>= {RARE_THRESHOLD} rows in train): {keep_districts}")

def collapse_district(series, keep_set):
    return series.apply(lambda x: x if x in keep_set else "Other")

for df in [train, cal, test]:
    df["homeDistrict"] = collapse_district(df["homeDistrict"], keep_districts)

print("\nDistricts after collapsing:", train["homeDistrict"].nunique())
print(train["homeDistrict"].value_counts())

Districts before collapsing: 23
Districts kept as own category (>= 30 rows in train): {'Kampala', 'Wakiso', 'Mukono', 'Kalangala'}

Districts after collapsing: 5
homeDistrict
Kampala      5719
Wakiso       2643
Other         134
Kalangala      62
Mukono         54
Name: count, dtype: int64


In [3]:
ug_out = run_base_learner_suite(
    train, cal, test, label_col="y_anxiety_binary",
    numeric_cols=NUMERIC_COLS, categorical_cols=CATEGORICAL_COLS,
    model_names=MODEL_NAMES, trial_budget=TRIAL_BUDGET, seed=SEED, n_bootstrap=N_BOOTSTRAP,
    models_dir=os.path.join(MODELS_ROOT, "uganda"), tables_dir=TABLES_DIR,
    dataset_tag="uganda", svm_subsample_n=SVM_FIT_ROWS,
)
ug_out["results_df"]

[uganda] Train/Cal/Test: (8612, 10)/(2872, 10)/(2841, 10) | unique training combos: 6440 of 8612
[uganda] dt: best CV AUC = 0.6554
[uganda] rf: best CV AUC = 0.6913
[uganda] gb: best CV AUC = 0.6900


C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `Calibra

[uganda] svm: best CV AUC = 0.6843
[uganda] nb: best CV AUC = 0.6485
[uganda] knn: best CV AUC = 0.6816
[uganda] xgb: best CV AUC = 0.6920


C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `Calibra

[uganda] Done. Results saved to C:\maternal-mental-health-risk\reports/tables, models saved to C:\maternal-mental-health-risk\models\uganda.


,model,threshold,accuracy,macro_f1,auc_roc,auc_ci_lower,auc_ci_upper,cv_auc_tuning
1,rf,0.55,0.6364,0.6314,0.6965,0.6770,0.7162,0.6913
6,xgb,0.54,0.6371,0.6340,0.6928,0.6732,0.7127,0.6920
2,gb,0.54,0.6290,0.6262,0.6912,0.6714,0.7106,0.6900
3,svm,0.48,0.6382,0.6379,0.6831,0.6626,0.7026,0.6843
5,knn,0.47,0.6283,0.6279,0.6786,0.6589,0.6978,0.6816
4,nb,0.12,0.6103,0.6071,0.6557,0.6354,0.6756,0.6485
0,dt,0.52,0.6199,0.6191,0.6511,0.6301,0.6720,0.6554


In [4]:
ug_out["oof_predictions"].to_csv(os.path.join(ug_dir, "oof_predictions_train.csv"), index=False)
ug_out["cal_predictions"].to_csv(os.path.join(ug_dir, "base_predictions_calibration.csv"), index=False)
ug_out["test_predictions"].to_csv(os.path.join(ug_dir, "base_predictions_test.csv"), index=False)
print("Saved.")

Saved.


In [5]:
corr_ug = ug_out["oof_predictions"][MODEL_NAMES].corr()
print(corr_ug.round(3))

        dt     rf     gb    svm     nb    knn    xgb
dt   1.000  0.795  0.796  0.717  0.445  0.715  0.810
rf   0.795  1.000  0.965  0.905  0.592  0.913  0.972
gb   0.796  0.965  1.000  0.914  0.615  0.879  0.984
svm  0.717  0.905  0.914  1.000  0.572  0.838  0.910
nb   0.445  0.592  0.615  0.572  1.000  0.507  0.630
knn  0.715  0.913  0.879  0.838  0.507  1.000  0.885
xgb  0.810  0.972  0.984  0.910  0.630  0.885  1.000


In [6]:
print("Total Score and Anxiety (continuous, 0-9 severity) are preserved in the processed")
print("Uganda files but not modeled in this notebook -- Phase 5 focuses on the classification")
print("task for comparability with Pakistan/Arab5. A supplementary regression analysis using")
print("the continuous outcome can be added later as an additional robustness angle if useful")
print("for the manuscript, using the same train/cal/test partitions.")

Total Score and Anxiety (continuous, 0-9 severity) are preserved in the processed
Uganda files but not modeled in this notebook -- Phase 5 focuses on the classification
task for comparability with Pakistan/Arab5. A supplementary regression analysis using
the continuous outcome can be added later as an additional robustness angle if useful
for the manuscript, using the same train/cal/test partitions.
